In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../db/rentals.db')

def q(sql):
    return pd.read_sql_query(sql, conn)

In [2]:
q("""
SELECT l.locality_name, l.locality_group, l.listing_count,
       ROUND(AVG(li.price * 1.0 / li.size_sq_ft), 2) AS avg_price_per_sqft,
       ROUND(AVG(li.price), 0) AS avg_price
FROM listings li
JOIN localities l ON li.locality_id = l.locality_id
WHERE l.is_sparse = 0
GROUP BY l.locality_name
ORDER BY avg_price_per_sqft DESC
LIMIT 20;
""")

,locality_name,locality_group,listing_count,avg_price_per_sqft,avg_price
0,Yojna Vihar,Delhi East,6,76.53,43333.0
1,Sukhdev Vihar,Delhi North,10,66.33,48300.0
2,Bank Enclave,Other,15,56.93,60767.0
3,Defence Colony,Delhi South,109,53.61,115242.0
4,Anand Niketan,Delhi South,10,52.62,142000.0
5,Neeti Bagh,Other,23,47.53,120778.0
6,Connaught Place,Delhi Central,9,47.43,78500.0
7,South Extension Part 1,Other,6,46.76,131219.0
8,Nizamuddin East,Delhi South,6,45.51,99833.0
9,South Extension 2,Delhi South,34,45.13,73176.0


In [3]:
q("""
SELECT li.listing_id, l.locality_name, l.locality_group, li.price,
       AVG(li.price) OVER (PARTITION BY l.locality_group) AS group_avg_price,
       ROUND(li.price - AVG(li.price) OVER (PARTITION BY l.locality_group), 0) AS diff_from_group_avg,
       NTILE(4) OVER (PARTITION BY l.locality_group ORDER BY li.price) AS price_quartile
FROM listings li
JOIN localities l ON li.locality_id = l.locality_id
ORDER BY diff_from_group_avg DESC
LIMIT 20;
""")

,listing_id,locality_name,locality_group,price,group_avg_price,diff_from_group_avg,price_quartile
0,7440,Prashant Vihar,West Delhi,220000,23504.130871,196496.0,4
1,8947,Ashok Vihar,North Delhi,220000,29082.704305,190917.0,4
2,15021,Neeti Bagh,Other,210000,25895.414130,184105.0,4
3,15865,Kotla Mubarakpur,Other,205000,25895.414130,179105.0,4
4,3390,Vasant Vihar,Delhi South,220000,43450.648693,176549.0,4
5,5842,Uttam Nagar,West Delhi,200000,23504.130871,176496.0,4
6,7278,Uttam Nagar,West Delhi,200000,23504.130871,176496.0,4
7,13721,Chanakyapuri,Delhi Central,200000,23866.865127,176133.0,4
8,14085,Tilak Marg,Delhi Central,200000,23866.865127,176133.0,4
9,14500,Neeti Bagh,Other,200000,25895.414130,174105.0,4


In [4]:
q("""
WITH group_stats AS (
    SELECT l.locality_group,
           COUNT(*) AS n_listings,
           AVG(li.price) AS avg_price,
           AVG(li.price * 1.0 / li.size_sq_ft) AS avg_price_per_sqft,
           AVG(li.closest_metro_km) AS avg_metro_dist
    FROM listings li
    JOIN localities l ON li.locality_id = l.locality_id
    GROUP BY l.locality_group
)
SELECT *,
       RANK() OVER (ORDER BY avg_price_per_sqft DESC) AS price_rank
FROM group_stats
ORDER BY price_rank;
""")

,locality_group,n_listings,avg_price,avg_price_per_sqft,avg_metro_dist,price_rank
0,Delhi West,264,31966.776515,39.064968,0.551765,1
1,Delhi North,169,28020.710059,30.330447,1.250510,2
2,Delhi South,3635,43450.648693,29.894944,1.006565,3
3,Delhi Central,2684,23866.865127,26.687815,0.997542,4
4,North West Delhi,10,26280.000000,26.538013,0.923024,5
5,South West Delhi,80,17593.600000,26.159688,0.555450,6
6,North Delhi,1231,29082.704305,26.122453,0.766088,7
7,Other,2562,25895.414130,25.602876,1.140735,8
8,Rohini,272,23958.823529,24.334054,1.342940,9
9,West Delhi,2193,23504.130871,21.768295,0.861817,10
